# Plan A: attention-enhanced YOLO26 for underwater object detection

Run the cells from top to bottom. Read `plans/01-underwater.md` first. Before you start:

1. Use **Runtime > Change runtime type** and pick a GPU (for example T4).
2. Get the Brackish dataset in YOLO format (see the `brackish` entry of `datasets/manifest.json`), zip it as `brackish.zip` and upload it to the top level of Google Drive. Once the team has a Hugging Face dataset repo, fill in the manifest and store the token as the Colab Secret `HF_TOKEN` instead.
3. Edit the **Configuration** cell: the variant, the seed and the number of epochs.

Expected dataset layout inside the zip (a Roboflow export with `train/`, `valid/`, `test/` folders is converted for you):

```
brackish/
  images/train   images/val   images/test
  labels/train   labels/val   labels/test
```

## 1. Install and check the GPU

In [ ]:
# Clone the project (or update it when the runtime is reused), install the pinned requirements and show the GPU
import os
import subprocess
import sys
from pathlib import Path
# Address of the team repository
repositoryUrl = "https://github.com/keithku321/object-detection"
# Folder on the Colab machine that holds the clone
repositoryFolder = Path("/content/objectDetection")
# A private repository needs a GitHub token stored as the Colab Secret GITHUB_TOKEN (key icon in the left bar); never paste a token into a cell
try:
    from google.colab import userdata
    githubToken = userdata.get("GITHUB_TOKEN")
except Exception:
    githubToken = None
cloneUrl = repositoryUrl.replace("https://", f"https://{githubToken}@") if githubToken else repositoryUrl
gitCommand = ["git", "-C", str(repositoryFolder), "pull", "--quiet"] if repositoryFolder.exists() else ["git", "clone", "--quiet", cloneUrl, str(repositoryFolder)]
gitResult = subprocess.run(gitCommand, capture_output=True, text=True)
# Print git's message with the token removed, so the token can never end up in a saved notebook
if gitResult.returncode != 0: print(gitResult.stderr.replace(githubToken or "GITHUB_TOKEN", "***"))
os.chdir(repositoryFolder)
if str(repositoryFolder) not in sys.path: sys.path.insert(0, str(repositoryFolder))
# Colab already has PyTorch; requirements.txt adds the pinned Ultralytics and the task libraries without touching it
!pip install -q -r requirements.txt
# Show the attached GPU; if this prints an error then change the runtime type to a GPU
!nvidia-smi
from common.environment import printEnvironment
printEnvironment()

## 2. Configuration

In [ ]:
# Name of this task; it names the results file and the run folders
taskName = "underwater"
# Dataset entry in datasets/manifest.json
datasetName = "brackish"
# Name of the zip file that you uploaded to the top level of your Google Drive
datasetZipName = "brackish.zip"
# Name of the folder that appears after unzipping, which must match the folder inside the zip
datasetFolderName = "brackish"
# Class names in the same order as the class numbers in your label files
classNames = ["crab", "fish", "jellyfish", "shrimp", "small_fish", "starfish"]
# Variant to train: baseline, yolo11, yolov8, or <block>-<placement> such as cbam-a, eca-a, simam-b, ca-c (see underwater/train.py)
variantName = "eca-a"
# Model scale; n is nano, s is small
modelScale = "n"
# Seed; exploration uses 0, the final runs use 0, 1 and 2
seed = 0
# Number of full passes over the training data; time one epoch first and set this from the compute budget (gate G1)
trainingEpochs = 50
# Image size in pixels; baseline and variants must use the same value
imageSize = 640
# Images per batch; 16 fits a free Colab T4 for nano and small models
batchSize = 16
# Drive folder for results, so that weights survive when Colab resets the runtime
driveResultsFolder = Path("/content/drive/MyDrive/yoloProjectResults")
# Local copy of the zip file on the Colab machine
localZipPath = Path("/content") / datasetZipName
# Local folder that will hold the unzipped dataset
datasetRootPath = Path("/content/datasets") / datasetFolderName

## 3. Mount Google Drive and unpack the dataset

The zip is copied to local storage first because training directly from Drive is slow. Its SHA-256 is compared with `datasets/manifest.json`; when no checksum is recorded yet, the real one is printed so you can add it.

In [ ]:
from google.colab import drive
# Mount Google Drive; a pop-up will ask for permission. Drive holds results and checkpoints only, never the working copy of the dataset
drive.mount("/content/drive")
driveResultsFolder.mkdir(parents=True, exist_ok=True)
# Bring back results rows of earlier sessions: a fresh clone only contains the rows that were committed
from common.results import syncResultFiles
print(syncResultFiles(taskName, driveResultsFolder / "results"), "results rows so far")

from common.datasets import convertRoboflowLayout, extractZip, fetchDataset, getManifestEntry, verifySha256
manifestEntry = getManifestEntry(datasetName)
if manifestEntry.get("huggingFaceRepo"):
    # The token comes from the Colab Secret HF_TOKEN; common/datasets.py reads it from the environment
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    fetchDataset(datasetName, "/content/datasets")
else:
    # Copy the zip to local storage because training straight from Drive is slow
    !cp "/content/drive/MyDrive/{datasetZipName}" "{localZipPath}"
    if verifySha256(localZipPath, manifestEntry.get("sha256")) is False: raise ValueError("The zip does not match the checksum in datasets/manifest.json")
    extractZip(localZipPath, "/content/datasets")
print("Converted Roboflow folders:", convertRoboflowLayout(datasetRootPath))

## 4. Helper functions

The logic lives in `common/` and `underwater/` so it can be tested on a PC; this cell only imports it and adds two display helpers.

In [ ]:
from IPython.display import display
from PIL import Image
# File endings that count as images
imageSuffixes = {".jpg", ".jpeg", ".png"}

# Display the first few images from a folder inside the notebook
def showSavedImages(folderPath, maximumCount=4):
    imageFiles = [file for file in sorted(Path(folderPath).iterdir()) if file.suffix.lower() in imageSuffixes]
    for imageFile in imageFiles[:maximumCount]:
        display(Image.open(imageFile))

# Save the results rows to Drive and print the summary table (mean ± std over seeds)
def saveAndShowResults():
    from common.results import syncResultFiles
    from common.summariseResults import writeSummary
    syncResultFiles(taskName, driveResultsFolder / "results")
    print(writeSummary() or "No results rows yet.")

from common.datasets import checkClassNumbers, checkSplit, writeDatasetYaml
from underwater.train import runVariant, taskName as moduleTaskName

## 5. Create dataset.yaml and sanity-check the data

Read any warnings carefully. Most training problems are really data problems.

In [ ]:
# Generate the dataset.yaml file from the class list in the configuration cell
hasTestSplit = (datasetRootPath / "images" / "test").exists()
datasetYamlPath = writeDatasetYaml(datasetRootPath, classNames, testSplit="images/test" if hasTestSplit else None)
for splitName in ["train", "val"] + (["test"] if hasTestSplit else []):
    checkSplit(datasetRootPath, splitName)
    checkClassNumbers(datasetRootPath, splitName, len(classNames))
print(datasetYamlPath.read_text())

## 6. Baseline

Train the unmodified `yolo26` model first (experiment U0). Every variant is compared with this run, so it uses the same data split, image size, epochs and seed. A finished baseline is skipped, so this cell is cheap to re-run.

In [ ]:
# Train and evaluate the plain pretrained model; runs are written to Drive and resume by themselves
baselineWeightsPath = runVariant("baseline", seed, trainingEpochs, datasetYamlPath, driveResultsFolder, modelScale, imageSize, batchSize)
saveAndShowResults()

## 7. Train

Results are written straight to Drive so the weights survive a disconnect. For an attention variant, read the log line **"Transferred N/M items from pretrained weights"**: only the new attention layers should be missing. If the count collapses, the comparison with the baseline is invalid.

In [ ]:
# Train and evaluate the variant chosen in the configuration cell, then log one row in results/underwater.csv
variantWeightsPath = runVariant(variantName, seed, trainingEpochs, datasetYamlPath, driveResultsFolder, modelScale, imageSize, batchSize)
saveAndShowResults()

### If Colab disconnected during training

Run the earlier setup cells (1 to 5) again, then run the cell below. It is the same call as the training cell: every run has a fixed folder on Drive, so training continues from the last saved epoch, and finished runs are skipped.

In [ ]:
# Continue the interrupted run from its last checkpoint on Drive (same call as above; finished runs are skipped)
# variantWeightsPath = runVariant(variantName, seed, trainingEpochs, datasetYamlPath, driveResultsFolder, modelScale, imageSize, batchSize)
# saveAndShowResults()

## 8. Evaluate the trained model

Accuracy is already logged by the training cells (on the NMS-free head; see `common/training.py`). This cell adds the speed protocol: batch 1, FP16, 640 px, 50 warm-up and 300 timed iterations. The real-time criterion of this plan is at least 30 FPS.

In [ ]:
from common.benchmark import benchmarkModel
from common.results import appendResultRow, buildResultRow
# Speed and size of the variant with the shared protocol; the row lands next to the accuracy row of the same variant
speedMetrics = benchmarkModel(variantWeightsPath)
print(speedMetrics)
appendResultRow(buildResultRow(taskName, f"yolo26{modelScale}", variantName, seed, {key: value for key, value in speedMetrics.items() if isinstance(value, (int, float))}, notes="speed protocol"))
print("Real-time criterion (30 FPS):", "met" if speedMetrics["fps"] >= 30 else "NOT met")
saveAndShowResults()

## 9. Look at real predictions

Scores hide failures, so look at the images, especially murky ones and small animals.

In [ ]:
from ultralytics import YOLO
from common.registerModules import registerCustomModules
from common.training import endToEndArguments
registerCustomModules()
# Predict on the validation images with the trained model and save annotated copies
trainedModel = YOLO(str(variantWeightsPath))
predictionResults = trainedModel.predict(source=str(datasetRootPath / "images" / "val"), save=True, conf=0.4, project="/content/runs/trained", name="predict", exist_ok=True, stream=False, **endToEndArguments)
showSavedImages("/content/runs/trained/predict", maximumCount=8)

## 10. Turbidity robustness (experiment U5)

This is the cell that turns the project into a finding: how fast does accuracy fall as the water gets murkier? Degraded copies of the **test** split are built at five severity levels (haze, red-channel loss, blur) and each model is scored on every level.

In [ ]:
from underwater.degrade import buildDegradedTestSets, evaluateTurbidityRobustness, plotRobustnessCurves
# Build the degraded test sets once; the recipe is deterministic, so every model sees the same images
robustnessSplit = "test" if hasTestSplit else "val"
yamlBySeverity = buildDegradedTestSets(datasetRootPath, "/content/datasets/brackish_turbidity", classNames, splitName=robustnessSplit)
robustnessCurves = {
    "baseline": evaluateTurbidityRobustness(baselineWeightsPath, yamlBySeverity, imageSize),
    variantName: evaluateTurbidityRobustness(variantWeightsPath, yamlBySeverity, imageSize),
}
for modelLabel, curve in robustnessCurves.items():
    print(modelLabel, {severity: round(score, 3) for severity, score in curve.items()})
display(Image.open(plotRobustnessCurves(robustnessCurves, driveResultsFolder / "underwater" / "turbidity_curves.png")))

## 11. Export and download (optional)

Download the weights for the demo app (`weights/` folder of the repository). Export only if you need another format.
The results rows now sit in `results/underwater.csv` inside the clone and in `yoloProjectResults/results/` on Drive. Rows are part of the repository: download the CSV (or copy it from Drive) and commit it on your branch, without weights. Strip the outputs of this notebook before committing it (**Edit > Clear all outputs**).

In [ ]:
from google.colab import files
# Download the final weights to your own computer; the file name follows <task>-<variant>-seed<k>.pt
files.download(str(variantWeightsPath))
# Export to ONNX for the deployment number; nms=False exports the NMS-free head
exportedModelPath = trainedModel.export(format="onnx", imgsz=imageSize, **endToEndArguments)
print(exportedModelPath)
saveAndShowResults()